# 04 · Reranking Analysis — `scale_75K` (75,000 papers)
Cross-encoder Precision@K with vs without reranking. Platform: Kaggle T4 — only if compute available

In [ ]:
# ══════════════════════════════════════════════════════════════
# SCALE EXPERIMENT CONFIG  ← only N_PAPERS and SCALE_LABEL change between tiers
# ══════════════════════════════════════════════════════════════
import os, sys, json, random
import numpy as np
import pandas as pd
import torch

SCALE_LABEL   = "scale_75K"        # identifier used in output filenames
N_PAPERS      = 75000              # ← THE ONLY NUMBER THAT CHANGES PER TIER
RANDOM_SEED   = 42               # NEVER change this
CHUNK_SIZE    = 400              # tokens
CHUNK_OVERLAP = 50               # tokens
RRF_K         = 60               # standard RRF parameter
TOP_K_STAGE1  = 50               # candidates sent to reranker
EVAL_K_VALUES = [1, 3, 5, 10, 20]

# ── Hardware (fixed by FIX 1) ─────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT    = "../../1_data"
RESULTS_DIR  = f"../../4_results/{SCALE_LABEL}"
GT_PATH      = f"{DATA_ROOT}/eval/ground_truth.json"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Scale : {SCALE_LABEL} | N = {N_PAPERS:,} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import pickle, numpy as np
from sentence_transformers import CrossEncoder, SentenceTransformer
from rank_bm25 import BM25Okapi
import json

df_chunks = pd.read_parquet(f"{RESULTS_DIR}/chunks.parquet")
texts = df_chunks['text'].tolist()
chunk_ids = df_chunks['chunk_id'].tolist()
cord_uids = df_chunks['cord_uid'].tolist()
id_to_cord = dict(zip(chunk_ids, cord_uids))
id_to_text = dict(zip(chunk_ids, texts))

with open(f"{RESULTS_DIR}/bm25_index.pkl", "rb") as f:
    bm25 = pickle.load(f)
embeddings = np.load(f"{RESULTS_DIR}/bge_m3_embeddings.npy")
model = SentenceTransformer("BAAI/bge-m3", device=DEVICE)

# Load ground truth
if os.path.exists(GT_PATH):
    with open(GT_PATH) as f:
        GROUND_TRUTH = json.load(f)
    print(f"Ground truth loaded: {len(GROUND_TRUTH)} queries")
else:
    print("WARNING: using pseudo-labels (run General/03_query_set.ipynb for fixed GT)")
    GROUND_TRUTH = None

with open("../../1_data/eval/queries.json") as f:
    QUERIES = json.load(f)

In [ ]:
# Load cross-encoder (MS MARCO — the domain-mismatch reranker we study)
print("Loading cross-encoder (ms-marco-MiniLM-L-6-v2)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Sanity check:", reranker.predict([("COVID-19 lung imaging", "CT scan ground glass opacity")]))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def hybrid_retrieve_full(query, k=TOP_K_STAGE1):
    q_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, embeddings)[0]
    dense_top = list(np.argsort(sims)[::-1][:k])
    tokens = query.lower().split()
    bm25_scores = bm25.get_scores(tokens)
    bm25_top = list(np.argsort(bm25_scores)[::-1][:k])
    rrf_scores = {}
    for lst in [dense_top, bm25_top]:
        for rank, idx in enumerate(lst):
            cid = chunk_ids[idx]
            rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0/(RRF_K + rank + 1)
    return sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k]

def rerank(query, candidate_ids):
    pairs = [(query, id_to_text[cid]) for cid in candidate_ids if cid in id_to_text]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidate_ids[:len(pairs)], scores), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in ranked]

In [ ]:
# Evaluate Precision@K with and without reranking
P_K_VALUES = [1, 3, 5, 10]

p_no_rerank = {k: [] for k in P_K_VALUES}
p_rerank    = {k: [] for k in P_K_VALUES}

for query in QUERIES:
    if GROUND_TRUTH and query not in GROUND_TRUTH:
        continue
    candidates = hybrid_retrieve_full(query, k=TOP_K_STAGE1)
    
    if GROUND_TRUTH:
        relevant_cords = set(GROUND_TRUTH[query])
    else:
        # Pseudo-GT fallback: top-3 from stage-1
        relevant_cords = {id_to_cord[c] for c in candidates[:3]}

    reranked = rerank(query, candidates)
    
    for k in P_K_VALUES:
        top_k_no  = {id_to_cord.get(c,"") for c in candidates[:k]}
        top_k_re  = {id_to_cord.get(c,"") for c in reranked[:k]}
        p_no_rerank[k].append(len(top_k_no & relevant_cords) / k)
        p_rerank[k].append(len(top_k_re & relevant_cords) / k)

print(f"Reranking impact — {SCALE_LABEL} ({len(QUERIES)} queries)")
print(f"{'K':>4}  {'No rerank':>10}  {'Reranked':>10}  {'Delta':>8}")
for k in P_K_VALUES:
    nr = np.mean(p_no_rerank[k])
    rr = np.mean(p_rerank[k])
    print(f"{k:>4}  {nr:>10.3f}  {rr:>10.3f}  {rr-nr:>+8.3f}")

In [ ]:
# Save results
rows = []
for k in P_K_VALUES:
    rows.append({"k": k, "p_no_rerank": np.mean(p_no_rerank[k]),
                 "p_rerank": np.mean(p_rerank[k])})
df_rerank = pd.DataFrame(rows)
df_rerank.to_csv(f"{RESULTS_DIR}/reranking_precision.csv", index=False)
print(f"Saved: {RESULTS_DIR}/reranking_precision.csv")
print(f"P@5 delta: {np.mean(p_rerank[5]) - np.mean(p_no_rerank[5]):+.3f}")
print("Negative = domain mismatch hurts | Positive = reranker helps")